In [1]:
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

In [2]:
from google.colab import files
uploaded = files.upload()

Saving quotes.txt.zip to quotes.txt.zip


In [3]:
!unzip -o /content/quotes.txt.zip -d /content/

# Verify that train.txt now exists
!ls -F /content/

Archive:  /content/quotes.txt.zip
  inflating: /content/quotes.csv     
quotes.csv  quotes.txt.zip  sample_data/


In [4]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a default padding token
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a default padding token
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [12]:
from datasets import load_dataset

def prepare_text_dataset_for_lm(file_path, tokenizer, block_size=128):
    # IMPORTANT: Ensure 'file_path' points to a plain text file, not a zip archive.
    # You need to manually unzip your zip file, e.g., to '/content/quotes.txt'.

    # Load the raw text dataset using the 'datasets' library
    # The 'text' builder can load a single text file.
    raw_datasets = load_dataset("csv", data_files=file_path, split="train")

    # Determine the name of the column containing the text
    # This assumes the text content is in the first column of the CSV.
    text_column_name = raw_datasets.column_names[0]

    # Tokenize the dataset
    def tokenize_function(examples):
        # examples[text_column_name] is a list of strings when batched=True.
        # Pass the list of strings directly to the tokenizer for batch processing.
        # truncation=True will truncate each individual sequence to max_length if it exceeds it.
        return tokenizer(examples[text_column_name], truncation=True, max_length=block_size)

    tokenized_datasets = raw_datasets.map(
        tokenize_function,
        batched=True,
        remove_columns=raw_datasets.column_names, # Remove all original columns
        desc="Running tokenizer on dataset",
    )

    # Group texts into fixed-size chunks (blocks) for language modeling
    # This function is adapted from Hugging Face examples for CLM
    def group_texts(examples):
        # Concatenate all texts.
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # We drop the small remainder to ensure all blocks are of 'block_size'.
        total_length = (total_length // block_size) * block_size
        # Split by chunks of block_size.
        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }
        # For causal language modeling, labels are just a copy of input_ids.
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(
        group_texts,
        batched=True,
        desc=f"Grouping texts in chunks of {block_size}",
    )

    return lm_datasets

train_dataset = prepare_text_dataset_for_lm(
    file_path="/content/quotes.csv", # REMINDER: Ensure you have unzipped train.txt.zip to this path
    tokenizer=tokenizer
)

Running tokenizer on dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Grouping texts in chunks of 128:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [14]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    warmup_steps=100,
    prediction_loss_only=True
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=36, training_loss=3.3054559495713978, metrics={'train_runtime': 14.9917, 'train_samples_per_second': 4.603, 'train_steps_per_second': 2.401, 'total_flos': 4507287552000.0, 'train_loss': 3.3054559495713978, 'epoch': 3.0})

In [16]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-finetuned/tokenizer_config.json', './gpt2-finetuned/tokenizer.json')

In [17]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer=tokenizer
)

prompt = "Happy"

output = generator(
    prompt,
    max_length=100,
    num_return_sequences=1,
    temperature=0.7
)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Happy.

The last time I went to a bar with a friend, I didn't go because I was bored. It was the first time I was even close to going. I was just trying to get something out of my life. The only way I'd get to work was to spend my days watching movies and playing video games. I couldn't make it past the first day of work and I just couldn't remember the last. I didn't want to be lonely, I wanted to be free. I didn't want to be a burden. I wanted to be an individual. I wanted to be the best person I could be.

I've been lucky to have friends who have never met me. I have so many people I can never really trust, never have been alone with, and I've been really lucky to have made a person who does not know me feel like he or she has been forgiven. I've been so lucky to have had people who are very much like me. I'm not going to give up. I've got to figure out what it is that makes me feel so special, and how to live without the hate, frustration, and fear that comes with being a single pa